# 🎯 Navigator-OptiLite — YOLO Detection
**AMD MI300X | ROCm | YOLOv8 | 3fps Webcam Stream**

Run cells top to bottom. Cell 4 starts the stream.

In [ ]:
# ── Cell 1: Imports & Device Check ──────────────────────────────────────────
import torch
import cv2
import numpy as np
import base64
import time
import threading
from PIL import Image
from io import BytesIO
from IPython.display import display, HTML, clear_output
from ultralytics import YOLO
import ipywidgets as widgets

# Check AMD GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: Running on CPU — check ROCm setup')

In [ ]:
# ── Cell 2: Color Map & Class Config ────────────────────────────────────────
# COCO class IDs → (R, G, B, label_override)
# We remap semantics beyond raw class names

# Tailwind-inspired hex → BGR for OpenCV
COLORS = {
    # Blues — humans
    'person':        (219, 130,  66),   # blue-500  (BGR)

    # Reds — dangerous animals
    'bear':          ( 30,  27, 127),   # red-900
    'elephant':      ( 30,  27, 127),
    'lion':          ( 30,  27, 127),   # not in COCO but future-safe

    # Orange — non-dangerous animals
    'dog':           ( 49, 130, 247),   # orange-400
    'cat':           ( 49, 130, 247),
    'horse':         ( 49, 130, 247),
    'cow':           ( 49, 130, 247),
    'sheep':         ( 49, 130, 247),
    'bird':          ( 49, 130, 247),

    # Greens — doors (YOLO detects under broader classes)
    # We'll detect via: 'door' in custom model, or remap 'refrigerator' etc.
    # For now using these as placeholders
    'door':          ( 22, 163,  74),   # green-600
    'car_door':      ( 74, 222, 128),   # green-300 (lighter)
    'bus_door':      ( 21, 128,  61),   # green-700 (darker)
    'house_door':    ( 20,  83,  45),   # green-900 (darkest)

    # Default fallback
    'default':       (200, 200, 200),   # gray
}

# COCO classes YOLO knows — map dangerous animals
DANGEROUS_ANIMALS = {'bear', 'elephant'}
SAFE_ANIMALS = {'dog', 'cat', 'horse', 'cow', 'sheep', 'bird', 'giraffe', 
                'zebra', 'sheep'}

def get_color(class_name: str) -> tuple:
    """Return BGR color for a given class name."""
    c = class_name.lower()
    if c in COLORS:
        return COLORS[c]
    if c in DANGEROUS_ANIMALS:
        return COLORS['bear']  # red-900
    if c in SAFE_ANIMALS:
        return COLORS['dog']   # orange
    if 'person' in c or 'human' in c:
        return COLORS['person']
    return COLORS['default']

print('Color map loaded ✓')

In [ ]:
# ── Cell 3: Load YOLO Model ──────────────────────────────────────────────────
print('Loading YOLOv8n (nano — fast for streaming)...')
model = YOLO('yolov8n.pt')  # auto-downloads on first run
model.to(device)
print(f'Model loaded on {device} ✓')
print(f'Classes: {len(model.names)} COCO classes')

In [ ]:
# ── Cell 4: Annotation Engine ────────────────────────────────────────────────
def draw_detections(frame: np.ndarray, results) -> np.ndarray:
    """
    Draw custom styled bounding boxes on frame.
    - Colored borders by semantic class
    - Rounded label pill
    - Corner-bracket style (not solid box) for cleaner look
    """
    annotated = frame.copy()
    
    for result in results:
        boxes = result.boxes
        if boxes is None:
            continue
            
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            class_name = model.names[cls_id]
            color = get_color(class_name)
            
            if conf < 0.4:  # skip low confidence
                continue
            
            # ── Corner bracket style ──────────────────────────
            bracket_len = min(30, (x2 - x1) // 4, (y2 - y1) // 4)
            thickness = 3
            
            # Top-left
            cv2.line(annotated, (x1, y1), (x1 + bracket_len, y1), color, thickness)
            cv2.line(annotated, (x1, y1), (x1, y1 + bracket_len), color, thickness)
            # Top-right
            cv2.line(annotated, (x2, y1), (x2 - bracket_len, y1), color, thickness)
            cv2.line(annotated, (x2, y1), (x2, y1 + bracket_len), color, thickness)
            # Bottom-left
            cv2.line(annotated, (x1, y2), (x1 + bracket_len, y2), color, thickness)
            cv2.line(annotated, (x1, y2), (x1, y2 - bracket_len), color, thickness)
            # Bottom-right
            cv2.line(annotated, (x2, y2), (x2 - bracket_len, y2), color, thickness)
            cv2.line(annotated, (x2, y2), (x2, y2 - bracket_len), color, thickness)
            
            # ── Label pill ────────────────────────────────────
            label = f'{class_name} {conf:.0%}'
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.55
            font_thickness = 1
            (tw, th), baseline = cv2.getTextSize(label, font, font_scale, font_thickness)
            
            pad = 5
            lx1, ly1 = x1, y1 - th - pad * 2 - baseline
            lx2, ly2 = x1 + tw + pad * 2, y1
            
            # keep label inside frame
            if ly1 < 0:
                ly1, ly2 = y2, y2 + th + pad * 2 + baseline
            
            # pill background
            cv2.rectangle(annotated, (lx1, ly1), (lx2, ly2), color, -1)
            
            # label text (white)
            cv2.putText(
                annotated, label,
                (lx1 + pad, ly2 - baseline - 1),
                font, font_scale, (255, 255, 255), font_thickness,
                cv2.LINE_AA
            )
    
    return annotated


def frame_to_b64(frame: np.ndarray) -> str:
    """Convert OpenCV frame → base64 JPEG string for HTML display."""
    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return base64.b64encode(buf).decode('utf-8')


print('Annotation engine ready ✓')

In [ ]:
# ── Cell 5: Webcam Capture Widget ────────────────────────────────────────────
# This injects JS into the notebook that:
# 1. Accesses your LOCAL webcam via browser API
# 2. Captures a frame every ~333ms (3fps)
# 3. Sends it as base64 to the Python kernel via a widget
# 4. Python runs YOLO → sends annotated frame back → displayed

# Widget to receive frames from JS
frame_widget = widgets.Text(value='', description='frame:', layout=widgets.Layout(display='none'))
display(frame_widget)

display(HTML('''
<style>
/* ── Navigator UI styles ── */
#nav-container {
    background: #0f0f0f;
    border-radius: 12px;
    padding: 16px;
    display: inline-block;
    font-family: 'Courier New', monospace;
}
#nav-header {
    color: #4ade80;
    font-size: 13px;
    margin-bottom: 10px;
    letter-spacing: 2px;
    text-transform: uppercase;
}
#nav-output {
    border-radius: 8px;
    border: 1px solid #222;
    overflow: hidden;
}
#nav-output img {
    display: block;
}
#nav-status {
    color: #666;
    font-size: 11px;
    margin-top: 8px;
    display: flex;
    justify-content: space-between;
}
#nav-fps { color: #4ade80; }
#nav-detections { color: #60a5fa; }
#nav-btn {
    background: #1a1a1a;
    border: 1px solid #333;
    color: #fff;
    padding: 6px 14px;
    border-radius: 6px;
    cursor: pointer;
    font-size: 12px;
    margin-bottom: 10px;
    transition: border-color 0.2s;
}
#nav-btn:hover { border-color: #4ade80; color: #4ade80; }
#nav-btn.active { border-color: #ef4444; color: #ef4444; }
</style>

<div id="nav-container">
    <div id="nav-header">⬡ Navigator-OptiLite // YOLO Detection</div>
    <button id="nav-btn" onclick="toggleStream()">▶ Start Stream</button>
    <div id="nav-output">
        <img id="nav-preview" width="640" height="480"
             src="data:image/gif;base64,R0lGODlhAQABAAAAACH5BAEKAAEALAAAAAABAAEAAAICTAEAOw=="
             style="background:#111;">
    </div>
    <div id="nav-status">
        <span>FPS: <span id="nav-fps">--</span></span>
        <span>Detections: <span id="nav-detections">--</span></span>
        <span id="nav-msg">Click Start to begin</span>
    </div>
</div>

<video id="nav-video" style="display:none" autoplay playsinline></video>
<canvas id="nav-canvas" style="display:none"></canvas>

<script>
let streamActive = false;
let videoStream = null;
let lastFrameTime = Date.now();
let frameCount = 0;

const TARGET_WIDTH = 640;
const TARGET_HEIGHT = 480;
const FPS = 3;

async function toggleStream() {
    const btn = document.getElementById('nav-btn');
    if (!streamActive) {
        try {
            videoStream = await navigator.mediaDevices.getUserMedia({
                video: { width: { ideal: 1280 }, height: { ideal: 720 }, facingMode: 'user' },
                audio: false
            });
            const video = document.getElementById('nav-video');
            video.srcObject = videoStream;
            await video.play();
            streamActive = true;
            btn.textContent = '⏹ Stop Stream';
            btn.classList.add('active');
            document.getElementById('nav-msg').textContent = 'Streaming...';
            captureLoop();
        } catch(e) {
            document.getElementById('nav-msg').textContent = 'Webcam error: ' + e.message;
        }
    } else {
        streamActive = false;
        if (videoStream) videoStream.getTracks().forEach(t => t.stop());
        btn.textContent = '▶ Start Stream';
        btn.classList.remove('active');
        document.getElementById('nav-msg').textContent = 'Stopped';
    }
}

function captureLoop() {
    if (!streamActive) return;
    const video = document.getElementById('nav-video');
    const canvas = document.getElementById('nav-canvas');
    canvas.width = TARGET_WIDTH;
    canvas.height = TARGET_HEIGHT;
    const ctx = canvas.getContext('2d');
    ctx.drawImage(video, 0, 0, TARGET_WIDTH, TARGET_HEIGHT);
    // JPEG at 80% quality — good balance for 3fps over loopback
    const b64 = canvas.toDataURL('image/jpeg', 0.8).split(',')[1];
    
    // Send to Python kernel via widget
    const widget = document.querySelector('input[data-widget-type="text"]') ||
                   document.querySelector('.widget-text input');
    if (widget) {
        // Trigger Jupyter widget update
        const nativeInputValueSetter = Object.getOwnPropertyDescriptor(
            window.HTMLInputElement.prototype, 'value').set;
        nativeInputValueSetter.call(widget, b64);
        widget.dispatchEvent(new Event('input', { bubbles: true }));
    }
    
    // FPS counter
    frameCount++;
    const now = Date.now();
    if (now - lastFrameTime > 1000) {
        document.getElementById('nav-fps').textContent = frameCount;
        frameCount = 0;
        lastFrameTime = now;
    }
    
    setTimeout(captureLoop, 1000 / FPS);
}

// Receive annotated frame back from Python and display it
window.showAnnotatedFrame = function(b64, detectionCount) {
    document.getElementById('nav-preview').src = 'data:image/jpeg;base64,' + b64;
    document.getElementById('nav-detections').textContent = detectionCount;
};
</script>
'''))
print('Webcam widget injected ✓')
print('Click the ▶ Start Stream button above to begin')

In [ ]:
# ── Cell 6: Python-side frame processor ──────────────────────────────────────
# This listens for frames sent by the JS widget,
# runs YOLO, sends annotated frame back to the browser display.

def process_frame(b64_str: str):
    """Decode base64 frame → YOLO → annotate → send back to browser."""
    try:
        # Decode JPEG
        img_bytes = base64.b64decode(b64_str)
        img_array = np.frombuffer(img_bytes, dtype=np.uint8)
        frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        if frame is None:
            return
        
        # Run YOLO
        results = model(frame, verbose=False, device=device)
        
        # Count detections
        detection_count = sum(len(r.boxes) for r in results if r.boxes)
        
        # Annotate
        annotated = draw_detections(frame, results)
        
        # Encode back to base64
        result_b64 = frame_to_b64(annotated)
        
        # Push back to browser via JS
        display(HTML(f'<script>window.showAnnotatedFrame("{result_b64}", {detection_count});</script>'))
        
    except Exception as e:
        print(f'Frame error: {e}')


# Hook into widget changes
def on_frame_received(change):
    val = change['new']
    if val and len(val) > 100:  # ignore empty/init
        process_frame(val)

frame_widget.observe(on_frame_received, names='value')

print('Frame processor ready ✓')
print('The notebook is now LIVE — press Start Stream in the widget above!')

In [ ]:
# ── Cell 7: Static image test (use before webcam to verify YOLO works) ────────
# Drop an image path here to test detection on a static file first

TEST_IMAGE = None  # Set to '/path/to/image.jpg' to test

if TEST_IMAGE:
    frame = cv2.imread(TEST_IMAGE)
    results = model(frame, verbose=True, device=device)
    annotated = draw_detections(frame, results)
    
    # Convert BGR→RGB for display
    rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    
    buf = BytesIO()
    pil_img.save(buf, format='JPEG', quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode()
    
    display(HTML(f'<img src="data:image/jpeg;base64,{b64}" style="max-width:800px;border-radius:8px;">'))
    
    det_count = sum(len(r.boxes) for r in results if r.boxes)
    print(f'Detected {det_count} objects')
else:
    print('Set TEST_IMAGE to a file path to test static detection')
    print('Or just run Cell 5 + 6 for live webcam stream')